In [9]:
# ============================================================
# MODULE 1: Data Pull + Closed-Form Tangency Portfolio
# ============================================================
import numpy as np
import pandas as pd
import yfinance as yf

# --- Same 20-ticker, 5-sector universe as the Stat Arb project ---
TICKERS = {
    'Tech':      ['AAPL', 'MSFT', 'NVDA', 'GOOGL'],
    'Financials':['JPM', 'BAC', 'MA', 'V'],
    'Energy':    ['XOM', 'CVX', 'COP', 'SLB'],
    'Retail':    ['WMT', 'TGT', 'HD', 'COST'],
    'Payments':  ['PYPL', 'SQ', 'FIS', 'GPN'],
}
ALL_TICKERS = [t for group in TICKERS.values() for t in group]
SECTOR_MAP = {t: sector for sector, group in TICKERS.items() for t in group}

RISK_FREE_RATE_ANNUAL = 0.04   # approx. current T-bill yield; adjust as needed
LOOKBACK_YEARS = 3

print(f"Pulling {len(ALL_TICKERS)} tickers, {LOOKBACK_YEARS}y lookback...")

raw = yf.download(ALL_TICKERS, period=f"{LOOKBACK_YEARS}y", auto_adjust=True)['Close']
raw = raw.dropna(axis=1, how='all')          # drop any ticker that failed entirely
raw = raw.dropna(axis=0, how='any')          # keep only dates with full data across all tickers

kept_tickers = list(raw.columns)
dropped = set(ALL_TICKERS) - set(kept_tickers)
if dropped:
    print(f"Warning: dropped tickers with no data: {dropped}")

daily_returns = raw.pct_change().dropna()

# --- Annualized inputs ---
mu = daily_returns.mean() * 252                      # annualized expected return vector
Sigma = daily_returns.cov() * 252                    # annualized covariance matrix

n = len(kept_tickers)
print(f"\nUsable universe: {n} tickers, {len(daily_returns)} trading days of returns")
print(f"Date range: {daily_returns.index[0].date()} to {daily_returns.index[-1].date()}")

# ============================================================
# Closed-form tangency (max-Sharpe) portfolio
# w ∝ Σ⁻¹(μ - rf·1), normalized to sum to 1
# ============================================================
excess_mu = mu.values - RISK_FREE_RATE_ANNUAL
Sigma_inv = np.linalg.inv(Sigma.values)

raw_weights = Sigma_inv @ excess_mu
print(f"Sum of raw weights before normalization: {raw_weights.sum():.6f}")
tangency_weights = raw_weights / raw_weights.sum()

tangency_weights = pd.Series(tangency_weights, index=kept_tickers)         # keep this order for all math
tangency_weights_display = tangency_weights.sort_values(ascending=False)    # this one is just for display

# --- Portfolio stats ---
port_return = tangency_weights.values @ mu.values
port_vol = np.sqrt(tangency_weights.values @ Sigma.values @ tangency_weights.values)
port_sharpe = (port_return - RISK_FREE_RATE_ANNUAL) / port_vol

print("\n=== Tangency (Max-Sharpe) Portfolio Weights ===")
print(tangency_weights_display.round(4))

print(f"\nPortfolio expected return (annualized): {port_return:.4f}")
print(f"Portfolio volatility (annualized):      {port_vol:.4f}")
print(f"Portfolio Sharpe ratio:                 {port_sharpe:.4f}")

# ============================================================
# Sanity check: does the tangency portfolio's Sharpe exceed
# every individual asset's own Sharpe ratio?
# ============================================================
individual_sharpe = (mu - RISK_FREE_RATE_ANNUAL) / np.sqrt(np.diag(Sigma))
individual_sharpe = individual_sharpe.sort_values(ascending=False)

print("\n=== Individual Asset Sharpe Ratios (sorted) ===")
print(individual_sharpe.round(4))

best_individual = individual_sharpe.max()
print(f"\nBest individual asset Sharpe: {best_individual:.4f}")
print(f"Tangency portfolio Sharpe:    {port_sharpe:.4f}")
print(f"Check passes (portfolio beats best individual asset): {port_sharpe > best_individual}")

Pulling 20 tickers, 3y lookback...


[*********************100%***********************]  20 of 20 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SQ']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')



Usable universe: 19 tickers, 750 trading days of returns
Date range: 2023-08-01 to 2026-07-28
Sum of raw weights before normalization: 11.834442

=== Tangency (Max-Sharpe) Portfolio Weights ===
XOM      0.8426
WMT      0.3885
JPM      0.3578
V        0.2592
GOOGL    0.2448
NVDA     0.2217
BAC      0.1729
AAPL     0.0854
TGT      0.0173
COST    -0.0001
CVX     -0.0582
PYPL    -0.0779
MA      -0.0825
GPN     -0.1329
HD      -0.1504
FIS     -0.1512
MSFT    -0.2139
SLB     -0.3093
COP     -0.4139
dtype: float64

Portfolio expected return (annualized): 0.6384
Portfolio volatility (annualized):      0.2249
Portfolio Sharpe ratio:                 2.6611

=== Individual Asset Sharpe Ratios (sorted) ===
Ticker
JPM      1.2312
NVDA     1.1785
WMT      1.1186
GOOGL    1.0548
BAC      0.9805
COST     0.8778
AAPL     0.6911
V        0.6636
XOM      0.6036
MA       0.5236
CVX      0.3266
TGT      0.2219
MSFT     0.2075
HD       0.0950
COP      0.0751
SLB     -0.0386
GPN     -0.1072
PYPL    -0.1259


In [10]:
# ============================================================
# DIAGNOSTIC: Is the covariance matrix ill-conditioned?
# ============================================================
cond_number = np.linalg.cond(Sigma.values)
print(f"Condition number of Sigma: {cond_number:.1f}")
print("(Rule of thumb: >1000 is concerning, >10,000 is severe ill-conditioning)")

# ============================================================
# Cross-check: find max-Sharpe portfolio via direct numerical
# optimization instead of matrix inversion, and compare
# ============================================================
from scipy.optimize import minimize

def neg_sharpe(w, mu, Sigma, rf):
    ret = w @ mu
    vol = np.sqrt(w @ Sigma @ w)
    return -(ret - rf) / vol

w0 = np.ones(n) / n  # equal-weight starting guess
constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]

result = minimize(
    neg_sharpe, w0, args=(mu.values, Sigma.values, RISK_FREE_RATE_ANNUAL),
    method='SLSQP', constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-12}
)

print(f"Numerical optimizer weight sum (should be ~1): {result.x.sum():.6f}")
print(f"Largest weight magnitude in numerical result: {np.max(np.abs(result.x)):.2f}")

numerical_sharpe = -result.fun
print(f"\nClosed-form tangency Sharpe: {port_sharpe:.4f}")
print(f"Numerical optimizer Sharpe:  {numerical_sharpe:.4f}")
print(f"Do they agree? {np.isclose(port_sharpe, numerical_sharpe, atol=1e-3)}")

Condition number of Sigma: 84.1
(Rule of thumb: >1000 is concerning, >10,000 is severe ill-conditioning)
Numerical optimizer weight sum (should be ~1): 1.000000
Largest weight magnitude in numerical result: 0.84

Closed-form tangency Sharpe: 2.6611
Numerical optimizer Sharpe:  2.6611
Do they agree? True


In [11]:
# ============================================================
# MODULE 2: Constrained Optimization
# (Leverage, Turnover, and Sector Limits)
# ============================================================
from scipy.optimize import minimize

# --- Constraint parameters (adjust these to test different scenarios) ---
MAX_LEVERAGE = 1.5      # sum of |weights| <= 1.5  (e.g. 150% gross exposure)
MAX_TURNOVER = 0.5      # sum of |w - w_previous| <= 0.5 (50% of portfolio can change)
MAX_SECTOR_WEIGHT = 0.35   # no sector's net weight can exceed 35%
MIN_SECTOR_WEIGHT = -0.35  # ...or go below -35% net short

# --- "Previous" portfolio for turnover purposes: assume starting equal-weight ---
w_previous = np.ones(n) / n

# --- Objective: same negative Sharpe as Module 1 ---
def neg_sharpe(w, mu, Sigma, rf):
    ret = w @ mu
    vol = np.sqrt(w @ Sigma @ w)
    return -(ret - rf) / vol

# --- Build sector membership matrix: rows = sectors, cols = tickers ---
sector_names = sorted(set(SECTOR_MAP.values()))
sector_matrix = np.zeros((len(sector_names), n))
for j, ticker in enumerate(kept_tickers):
    sector = SECTOR_MAP[ticker]
    i = sector_names.index(sector)
    sector_matrix[i, j] = 1

# --- Constraints ---
constraints = [
    # 1. Weights must sum to 1 (fully invested)
    {'type': 'eq', 'fun': lambda w: w.sum() - 1},

    # 2. Leverage: sum(|w|) <= MAX_LEVERAGE
    {'type': 'ineq', 'fun': lambda w: MAX_LEVERAGE - np.sum(np.abs(w))},

    # 3. Turnover: sum(|w - w_previous|) <= MAX_TURNOVER
    {'type': 'ineq', 'fun': lambda w: MAX_TURNOVER - np.sum(np.abs(w - w_previous))},
]

# 4. Sector limits: one constraint pair per sector (upper and lower bound)
for i, sector in enumerate(sector_names):
    constraints.append({
        'type': 'ineq',
        'fun': (lambda w, i=i: MAX_SECTOR_WEIGHT - sector_matrix[i] @ w)
    })
    constraints.append({
        'type': 'ineq',
        'fun': (lambda w, i=i: sector_matrix[i] @ w - MIN_SECTOR_WEIGHT)
    })

# --- Run the constrained optimizer ---
w0 = np.ones(n) / n  # equal-weight starting guess

result_constrained = minimize(
    neg_sharpe, w0, args=(mu.values, Sigma.values, RISK_FREE_RATE_ANNUAL),
    method='SLSQP', constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-12}
)

constrained_weights = pd.Series(result_constrained.x, index=kept_tickers)
constrained_weights_display = constrained_weights.sort_values(ascending=False)

constrained_sharpe = -result_constrained.fun
constrained_return = constrained_weights.values @ mu.values
constrained_vol = np.sqrt(constrained_weights.values @ Sigma.values @ constrained_weights.values)

print("=== Constrained Portfolio Weights ===")
print(constrained_weights_display.round(4))

print(f"\nConstrained portfolio expected return: {constrained_return:.4f}")
print(f"Constrained portfolio volatility:      {constrained_vol:.4f}")
print(f"Constrained portfolio Sharpe:           {constrained_sharpe:.4f}")

# --- Compare against unconstrained Module 1 result ---
print(f"\nUnconstrained (Module 1) Sharpe: {port_sharpe:.4f}")
print(f"Constrained (Module 2) Sharpe:   {constrained_sharpe:.4f}")
print(f"Sharpe given up due to constraints: {port_sharpe - constrained_sharpe:.4f}")

# --- Sanity checks: did the optimizer actually respect the constraints? ---
gross_leverage = np.sum(np.abs(constrained_weights.values))
turnover = np.sum(np.abs(constrained_weights.values - w_previous))

print(f"\n--- Constraint checks ---")
print(f"Gross leverage used: {gross_leverage:.4f}  (limit: {MAX_LEVERAGE})  OK: {gross_leverage <= MAX_LEVERAGE + 1e-6}")
print(f"Turnover used:       {turnover:.4f}  (limit: {MAX_TURNOVER})  OK: {turnover <= MAX_TURNOVER + 1e-6}")

print("\nSector net weights:")
for i, sector in enumerate(sector_names):
    sector_weight = sector_matrix[i] @ constrained_weights.values
    within_bounds = MIN_SECTOR_WEIGHT - 1e-6 <= sector_weight <= MAX_SECTOR_WEIGHT + 1e-6
    print(f"  {sector:12s}: {sector_weight:+.4f}  (limits: {MIN_SECTOR_WEIGHT} to {MAX_SECTOR_WEIGHT})  OK: {within_bounds}")

=== Constrained Portfolio Weights ===
WMT      0.1691
NVDA     0.1333
GOOGL    0.0820
JPM      0.0761
BAC      0.0526
V        0.0526
COST     0.0526
XOM      0.0526
MA       0.0526
CVX      0.0526
AAPL     0.0526
MSFT     0.0526
HD       0.0526
COP      0.0526
TGT      0.0526
FIS      0.0526
SLB     -0.0074
GPN     -0.0407
PYPL    -0.0440
dtype: float64

Constrained portfolio expected return: 0.2683
Constrained portfolio volatility:      0.1450
Constrained portfolio Sharpe:           1.5751

Unconstrained (Module 1) Sharpe: 2.6611
Constrained (Module 2) Sharpe:   1.5751
Sharpe given up due to constraints: 1.0860

--- Constraint checks ---
Gross leverage used: 1.1842  (limit: 1.5)  OK: True
Turnover used:       0.5000  (limit: 0.5)  OK: True

Sector net weights:
  Energy      : +0.1505  (limits: -0.35 to 0.35)  OK: True
  Financials  : +0.2340  (limits: -0.35 to 0.35)  OK: True
  Payments    : -0.0321  (limits: -0.35 to 0.35)  OK: True
  Retail      : +0.3270  (limits: -0.35 to 0.35)  

In [12]:
# ============================================================
# DIAGNOSTIC: Re-run from a different starting point
# ============================================================
np.random.seed(42)
w0_random = np.random.uniform(-0.1, 0.1, n)
w0_random = w0_random / w0_random.sum()  # normalize to sum to 1

result_check = minimize(
    neg_sharpe, w0_random, args=(mu.values, Sigma.values, RISK_FREE_RATE_ANNUAL),
    method='SLSQP', constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-12}
)

check_sharpe = -result_check.fun
print(f"Original run Sharpe (from equal-weight start): {constrained_sharpe:.4f}")
print(f"New run Sharpe (from random start):            {check_sharpe:.4f}")
print(f"Do they agree? {np.isclose(constrained_sharpe, check_sharpe, atol=1e-3)}")

Original run Sharpe (from equal-weight start): 1.5751
New run Sharpe (from random start):            1.5751
Do they agree? True


In [13]:
# ============================================================
# MODULE 3: Estimation Risk & Covariance Shrinkage
# ============================================================

# ------------------------------------------------------------
# Part A: Is the sample covariance matrix actually stable?
# Test: split the 750-day history into two halves, estimate
# Sigma separately on each half, and compare the resulting
# tangency weights. If they swing wildly, that's estimation
# risk showing up concretely, not just in theory.
# ------------------------------------------------------------
half = len(daily_returns) // 2
returns_first_half = daily_returns.iloc[:half]
returns_second_half = daily_returns.iloc[half:]

mu_1 = returns_first_half.mean() * 252
Sigma_1 = returns_first_half.cov() * 252
mu_2 = returns_second_half.mean() * 252
Sigma_2 = returns_second_half.cov() * 252

def tangency_weights_from(mu_vec, Sigma_mat, rf):
    excess = mu_vec.values - rf
    raw = np.linalg.inv(Sigma_mat.values) @ excess
    return raw / raw.sum()

w_first = pd.Series(tangency_weights_from(mu_1, Sigma_1, RISK_FREE_RATE_ANNUAL), index=kept_tickers)
w_second = pd.Series(tangency_weights_from(mu_2, Sigma_2, RISK_FREE_RATE_ANNUAL), index=kept_tickers)

weight_diff = (w_first - w_second).abs()

print("=== Part A: Stability Check (first half vs. second half of history) ===")
print(f"Trading days per half: {half}")
print(f"\nAverage absolute weight difference between halves: {weight_diff.mean():.4f}")
print(f"Largest single weight swing: {weight_diff.max():.4f} (ticker: {weight_diff.idxmax()})")
print("\nSide-by-side (first half vs second half), sorted by biggest swing:")
comparison = pd.DataFrame({'First Half': w_first, 'Second Half': w_second, 'Abs Diff': weight_diff})
print(comparison.sort_values('Abs Diff', ascending=False).round(4))

# ------------------------------------------------------------
# Part B: Ledoit-Wolf shrinkage
# Instead of trusting the raw sample covariance matrix,
# "shrink" it toward a simpler, more stable structured target
# (average correlation model), blending real data with a
# stable assumption to reduce estimation noise.
# ------------------------------------------------------------
from sklearn.covariance import LedoitWolf

lw = LedoitWolf()
lw.fit(daily_returns.values)
Sigma_shrunk = pd.DataFrame(lw.covariance_ * 252, index=kept_tickers, columns=kept_tickers)

print(f"\n=== Part B: Ledoit-Wolf Shrinkage ===")
print(f"Shrinkage intensity applied: {lw.shrinkage_:.4f}  (0 = no shrinkage, 1 = fully shrunk to target)")

# Re-run the SAME stability check using shrunk covariance instead of sample covariance
Sigma_1_shrunk = pd.DataFrame(LedoitWolf().fit(returns_first_half.values).covariance_ * 252,
                                index=kept_tickers, columns=kept_tickers)
Sigma_2_shrunk = pd.DataFrame(LedoitWolf().fit(returns_second_half.values).covariance_ * 252,
                                index=kept_tickers, columns=kept_tickers)

w_first_shrunk = pd.Series(tangency_weights_from(mu_1, Sigma_1_shrunk, RISK_FREE_RATE_ANNUAL), index=kept_tickers)
w_second_shrunk = pd.Series(tangency_weights_from(mu_2, Sigma_2_shrunk, RISK_FREE_RATE_ANNUAL), index=kept_tickers)

weight_diff_shrunk = (w_first_shrunk - w_second_shrunk).abs()

print(f"\nAverage absolute weight difference between halves (SHRUNK): {weight_diff_shrunk.mean():.4f}")
print(f"Largest single weight swing (SHRUNK): {weight_diff_shrunk.max():.4f} (ticker: {weight_diff_shrunk.idxmax()})")

print(f"\n--- Did shrinkage actually help? ---")
print(f"Average weight swing WITHOUT shrinkage: {weight_diff.mean():.4f}")
print(f"Average weight swing WITH shrinkage:    {weight_diff_shrunk.mean():.4f}")
improvement = (1 - weight_diff_shrunk.mean() / weight_diff.mean()) * 100
print(f"Improvement: {improvement:.1f}% reduction in weight instability")

=== Part A: Stability Check (first half vs. second half of history) ===
Trading days per half: 375

Average absolute weight difference between halves: 0.3630
Largest single weight swing: 1.3872 (ticker: XOM)

Side-by-side (first half vs second half), sorted by biggest swing:
       First Half  Second Half  Abs Diff
XOM        0.3792       1.7664    1.3872
COP       -0.0819      -0.9944    0.9125
FIS        0.0899      -0.6817    0.7717
HD         0.0676      -0.6925    0.7601
GOOGL      0.1482       0.5871    0.4388
COST       0.0919      -0.3223    0.4142
TGT       -0.0636       0.3123    0.3759
V          0.1859       0.5491    0.3632
BAC        0.0503       0.3703    0.3200
WMT        0.5272       0.2328    0.2944
AAPL       0.0227       0.2980    0.2753
SLB       -0.3650      -0.1571    0.2079
JPM        0.4252       0.2916    0.1336
GPN       -0.1998      -0.1292    0.0706
MA         0.0133      -0.0528    0.0661
PYPL      -0.0698      -0.1254    0.0555
MSFT      -0.2327      -0.2

In [14]:
# ============================================================
# MODULE 4: Walk-Forward Backtest
# (Constrained vs. Unconstrained, with transaction costs)
# ============================================================

# --- Backtest parameters ---
ESTIMATION_WINDOW = 252   # trading days used to estimate mu/Sigma before each rebalance
REBALANCE_FREQUENCY = 21  # rebalance roughly monthly (~21 trading days)
TRANSACTION_COST_BPS = 10  # 10 basis points (0.10%) charged per unit of turnover

dates = daily_returns.index
n_days = len(dates)

# Storage for results
portfolio_value_constrained = [1.0]
portfolio_value_unconstrained = [1.0]
weights_constrained_prev = np.ones(n) / n   # start equal-weight
weights_unconstrained_prev = np.ones(n) / n

rebalance_dates = []
turnover_log_constrained = []
turnover_log_unconstrained = []

# --- Walk forward, one rebalance at a time, NO LOOKAHEAD ---
# At each rebalance point t: estimate mu/Sigma using only data BEFORE t,
# hold those weights until the NEXT rebalance point, then repeat.
start = ESTIMATION_WINDOW
t = start

while t + REBALANCE_FREQUENCY < n_days:
    # --- Estimate mu/Sigma using only past data (no lookahead) ---
    window_returns = daily_returns.iloc[t - ESTIMATION_WINDOW:t]
    mu_t = window_returns.mean() * 252
    Sigma_t = pd.DataFrame(LedoitWolf().fit(window_returns.values).covariance_ * 252,
                            index=kept_tickers, columns=kept_tickers)

    # --- Unconstrained (closed-form tangency) ---
    w_unconstrained = tangency_weights_from(mu_t, Sigma_t, RISK_FREE_RATE_ANNUAL)

    # --- Constrained (same constraints as Module 2, applied fresh each rebalance) ---
    constraints_t = [
        {'type': 'eq', 'fun': lambda w: w.sum() - 1},
        {'type': 'ineq', 'fun': lambda w: MAX_LEVERAGE - np.sum(np.abs(w))},
        {'type': 'ineq', 'fun': lambda w: MAX_TURNOVER - np.sum(np.abs(w - weights_constrained_prev))},
    ]
    for i, sector in enumerate(sector_names):
        constraints_t.append({'type': 'ineq', 'fun': (lambda w, i=i: MAX_SECTOR_WEIGHT - sector_matrix[i] @ w)})
        constraints_t.append({'type': 'ineq', 'fun': (lambda w, i=i: sector_matrix[i] @ w - MIN_SECTOR_WEIGHT)})

    result_t = minimize(
        neg_sharpe, weights_constrained_prev, args=(mu_t.values, Sigma_t.values, RISK_FREE_RATE_ANNUAL),
        method='SLSQP', constraints=constraints_t, options={'maxiter': 1000, 'ftol': 1e-10}
    )
    w_constrained = result_t.x

    # --- Transaction costs: charged only on the CHANGE in weights (turnover) ---
    turnover_c = np.sum(np.abs(w_constrained - weights_constrained_prev))
    turnover_u = np.sum(np.abs(w_unconstrained - weights_unconstrained_prev))
    cost_c = turnover_c * (TRANSACTION_COST_BPS / 10000)
    cost_u = turnover_u * (TRANSACTION_COST_BPS / 10000)

    turnover_log_constrained.append(turnover_c)
    turnover_log_unconstrained.append(turnover_u)
    rebalance_dates.append(dates[t])

    # --- Hold these weights until the next rebalance, applying realized returns ---
    holding_period_returns = daily_returns.iloc[t:t + REBALANCE_FREQUENCY]

    period_return_c = (holding_period_returns.values @ w_constrained).sum() - cost_c
    period_return_u = (holding_period_returns.values @ w_unconstrained).sum() - cost_u

    portfolio_value_constrained.append(portfolio_value_constrained[-1] * (1 + period_return_c))
    portfolio_value_unconstrained.append(portfolio_value_unconstrained[-1] * (1 + period_return_u))

    weights_constrained_prev = w_constrained
    weights_unconstrained_prev = w_unconstrained
    t += REBALANCE_FREQUENCY

# ============================================================
# Results
# ============================================================
final_value_c = portfolio_value_constrained[-1]
final_value_u = portfolio_value_unconstrained[-1]

total_return_c = (final_value_c - 1) * 100
total_return_u = (final_value_u - 1) * 100

print(f"Number of rebalances: {len(rebalance_dates)}")
print(f"Backtest period: {rebalance_dates[0].date()} to {dates[-1].date()}")

print(f"\n=== Final Results ===")
print(f"Constrained strategy total return:   {total_return_c:+.2f}%")
print(f"Unconstrained strategy total return: {total_return_u:+.2f}%")

print(f"\nAverage turnover per rebalance (constrained):   {np.mean(turnover_log_constrained):.4f}")
print(f"Average turnover per rebalance (unconstrained): {np.mean(turnover_log_unconstrained):.4f}")

print(f"\nTotal transaction cost drag (constrained):   {sum(turnover_log_constrained) * TRANSACTION_COST_BPS / 10000 * 100:.2f}%")
print(f"Total transaction cost drag (unconstrained): {sum(turnover_log_unconstrained) * TRANSACTION_COST_BPS / 10000 * 100:.2f}%")

# --- Simple realized Sharpe over the backtest period, for both ---
returns_c = np.diff(portfolio_value_constrained) / portfolio_value_constrained[:-1]
returns_u = np.diff(portfolio_value_unconstrained) / portfolio_value_unconstrained[:-1]

periods_per_year = 252 / REBALANCE_FREQUENCY
sharpe_c = (np.mean(returns_c) * periods_per_year) / (np.std(returns_c) * np.sqrt(periods_per_year))
sharpe_u = (np.mean(returns_u) * periods_per_year) / (np.std(returns_u) * np.sqrt(periods_per_year))

print(f"\nRealized Sharpe (constrained):   {sharpe_c:.4f}")
print(f"Realized Sharpe (unconstrained): {sharpe_u:.4f}")

Number of rebalances: 23
Backtest period: 2024-08-01 to 2026-07-28

=== Final Results ===
Constrained strategy total return:   +62.14%
Unconstrained strategy total return: +111.66%

Average turnover per rebalance (constrained):   0.4906
Average turnover per rebalance (unconstrained): 2.9492

Total transaction cost drag (constrained):   1.13%
Total transaction cost drag (unconstrained): 6.78%

Realized Sharpe (constrained):   1.6292
Realized Sharpe (unconstrained): 1.0592
